## Whaat this Module Does:


1.   Receives a pipeline result dict (safe/unsafe) from the VLM
2.   Generates a text-based console alert with full threat reasoning
3. Saves an event log (JSON-Lines): timestamp, frame index, bounding boxes, objects, description
4. Saves an annotated JPEG frame
5. Exposes process_pipeline_result() — the single function your friend calls















## Import and Configuration

In [ ]:
from pathlib import Path

try:
    import cv2
    import numpy as np
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False
    print('[WARN] opencv-python not found — frame annotation disabled.')
    print('       pip install opencv-python')

# Edit these paths to match your environment
CONFIG = {
    'output_dir'                : './alert_output',
    'log_file'                  : './alert_output/safety_events.log',
    'frames_dir'                : './alert_output/frames',
    'alert_confidence_threshold': 0.20,
    'log_safe_scenes'           : False,
    'jpeg_quality'              : 90,
}

Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['frames_dir']).mkdir(parents=True, exist_ok=True)
print('Configuration loaded.')
print(f"  Event log  -> {CONFIG['log_file']}")
print(f"  Frames dir -> {CONFIG['frames_dir']}")

Configuration loaded.
  Event log  -> ./alert_output/safety_events.log
  Frames dir -> ./alert_output/frames


## EventLogger

In [ ]:
import hashlib
import time
import datetime
import json
from typing import Dict, List, Any, Optional

class EventLogger:
    """
    Appends one JSON record per unsafe event to the log file.
    Schema (SDD Section 2.4.1):
      event_id, session_id, timestamp_iso, timestamp_epoch,
      environment, event_label, scene_description, confidence,
      detected_objects, scene_graph_nodes, scene_graph_edges,
      contributing_nodes, frame_path, video_source, frame_index,
      threat_reasoning
    """

    def __init__(self, log_file: str):
        self.log_file = log_file

    def _eid(self, ts: float, sid: str) -> str:
        return hashlib.md5(f'{sid}-{ts}'.encode()).hexdigest()[:12]

    def write(
        self, session_id, environment, event_label, scene_description,
        confidence, detected_objects, scene_graph_nodes, scene_graph_edges,
        contributing_nodes, frame_path, video_source, frame_index, threat_reasoning
    ) -> Dict:
        now = time.time()
        iso = datetime.datetime.utcfromtimestamp(now).strftime('%Y-%m-%dT%H:%M:%S.%f') + 'Z'
        record = {
            'event_id'          : self._eid(now, session_id),
            'session_id'        : session_id,
            'timestamp_iso'     : iso,
            'timestamp_epoch'   : now,
            'environment'       : environment,
            'event_label'       : event_label,
            'scene_description' : scene_description,
            'confidence'        : round(confidence, 4),
            'detected_objects'  : detected_objects,
            'scene_graph_nodes' : scene_graph_nodes,
            'scene_graph_edges' : scene_graph_edges,
            'contributing_nodes': contributing_nodes,
            'frame_path'        : frame_path,
            'video_source'      : video_source,
            'frame_index'       : frame_index,
            'threat_reasoning'  : threat_reasoning,
        }
        with open(self.log_file, 'a', encoding='utf-8') as f:
            f.write(json.dumps(record) + '\n')
        return record


_event_logger = EventLogger(CONFIG['log_file'])
print('EventLogger ready.')

EventLogger ready.


## FrameAnnotator

In [ ]:
class FrameAnnotator:
    """Annotates a cv2 BGR frame and saves a JPEG to disk."""

    COLOURS = {
        'gun'    : (0,   0,   255),
        'knife'  : (0,   128, 255),
        'fire'   : (0,   69,  255),
        'fall'   : (255, 0,   255),
        'person' : (0,   255, 0),
        'default': (255, 255, 0),
    }

    def _pick_colour(self, label: str):
        ll = label.lower()
        for k, v in self.COLOURS.items():
            if k in ll:
                return v
        return self.COLOURS['default']

    def annotate_and_save(
        self, frame, detected_objects, event_label,
        environment, confidence, event_id, frames_dir, jpeg_quality=90
    ) -> Optional[str]:
        if not CV2_AVAILABLE:
            return None
        if frame is None:
            frame = np.zeros((480, 640, 3), dtype=np.uint8)
        ann = frame.copy()
        h, w = ann.shape[:2]

        for obj in detected_objects:
            box  = obj.get('bounding_box')
            lbl  = obj.get('label', '?')
            conf = obj.get('confidence', 0.0)
            if box and len(box) == 4:
                x1, y1, x2, y2 = [int(v) for v in box]
                col = self._pick_colour(lbl)
                cv2.rectangle(ann, (x1, y1), (x2, y2), col, 2)
                tag = f'{lbl} {conf:.0%}'
                (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
                cv2.rectangle(ann, (x1, y1 - th - 6), (x1 + tw + 4, y1), col, -1)
                cv2.putText(ann, tag, (x1 + 2, y1 - 4),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1)

        banner = (f'UNSAFE | {event_label} | ENV: {environment.upper()} '
                  f'| CONF: {confidence:.0%} | ID: {event_id}')
        cv2.rectangle(ann, (0, h - 50), (w, h), (0, 0, 180), -1)
        cv2.putText(ann, banner, (8, h - 16),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1)

        save_path = str(Path(frames_dir) / f'{event_id}.jpg')
        cv2.imwrite(save_path, ann, [cv2.IMWRITE_JPEG_QUALITY, jpeg_quality])
        return save_path


_frame_annotator = FrameAnnotator()
print('FrameAnnotator ready.')

FrameAnnotator ready.


## Threat Reaasoning Engine

In [ ]:
ENVIRONMENT_RULES: Dict[str, List[str]] = {
    'home'     : ['gun','knife','fire','flames','smoke','burning','person falling','fall'],
    'office'   : ['gun','fire','flames','smoke','person falling','fall'],
    'classroom': ['gun','knife','fire','flames','smoke','burning','person falling','fall'],
}

# Maps detected class names -> canonical alert label
EVENT_LABEL_MAP: Dict[str, str] = {
    'gun'           : 'Gun Detection',
    'knife'         : 'Knife Detection',
    'fire'          : 'Fire Detection',
    'flames'        : 'Fire Detection',
    'smoke'         : 'Fire Detection',
    'burning'       : 'Fire Detection',
    'person falling': 'Fall Detection',
    'fall'          : 'Fall Detection',
}


def build_threat_reasoning(
    environment: str,
    detected_objects: List[Dict],
    event_label: str,
    scene_description: str,
) -> str:
    """
    Returns a plain-English explanation of why the scene is UNSAFE.
    Covers: detected objects, environment context, rule applied, VLM description.
    """
    obj_summaries = ', '.join(
        f"{o['label']} ({o.get('confidence', 0):.0%})" for o in detected_objects
    ) or '(no individual objects reported)'

    rule_note = {
        'Gun Detection'  : f'Firearms are never permitted in a {environment.upper()} environment.',
        'Knife Detection': (
            'Knife detected outside kitchen context — potential threat.'
            if environment == 'home'
            else f'Bladed weapons are not permitted in a {environment.upper()} environment.'
        ),
        'Fire Detection' : f'Fire, flames, or smoke detected in {environment.upper()} — immediate hazard.',
        'Fall Detection' : 'A person appears to have fallen and may be injured.',
    }.get(event_label, f'Unsafe condition detected in {environment.upper()} environment.')

    return (
        f'Environment: {environment.upper()}\n'
        f'Detected objects: {obj_summaries}\n'
        f'Threat type: {event_label}\n'
        f'Rule applied: {rule_note}\n'
        f'Scene context: {scene_description.strip()}'
    )


def infer_event_label_from_objects(
    detected_objects: List[Dict], environment: str
) -> Optional[str]:
    """
    Fallback: derive event_label from detected_objects using environment rules.
    Returns canonical event label, or None if no unsafe object matches.
    """
    rules = ENVIRONMENT_RULES.get(environment.lower(), [])
    for obj in detected_objects:
        ll = obj.get('label', '').lower()
        for trigger in rules:
            if trigger in ll:
                return EVENT_LABEL_MAP.get(trigger, 'Unsafe Event')
    return None


print('Threat reasoning engine loaded.')

Threat reasoning engine loaded.


## Console Alert Formatter

In [ ]:
BORDER = '=' * 70
THIN   = '-' * 70


def print_unsafe_alert(
    event_id, timestamp_iso, environment, event_label, confidence,
    threat_reasoning, scene_description, detected_objects,
    frame_path, video_source, frame_index
) -> None:
    print()
    print(BORDER)
    print(f'UNSAFE SCENE DETECTED  --  {event_label.upper()}')
    print(BORDER)
    print(f'  Event ID     : {event_id}')
    print(f'  Timestamp    : {timestamp_iso}')
    print(f'  Environment  : {environment.upper()}')
    print(f'  Threat Type  : {event_label}')
    print(f'  Confidence   : {confidence:.1%}')
    print(f'  Video Source : {video_source}')
    print(f'  Frame Index  : {frame_index}')
    print(THIN)
    print('  DETECTED OBJECTS:')
    if detected_objects:
        for obj in detected_objects:
            box  = obj.get('bounding_box')
            bbox = f'[{box[0]:.0f},{box[1]:.0f},{box[2]:.0f},{box[3]:.0f}]' if box else 'N/A'
            print(f"    * {obj.get('label','?'):22s}  conf={obj.get('confidence',0):.0%}  bbox={bbox}")
    else:
        print('    (no individual object detections provided)')
    print(THIN)
    print('  THREAT REASONING:')
    for line in threat_reasoning.splitlines():
        print(f'    {line}')
    print(THIN)
    print('  SCENE DESCRIPTION:')
    desc = scene_description.strip()
    for i in range(0, max(len(desc), 1), 65):
        print(f'    {desc[i:i+65]}')
    print(THIN)
    if frame_path:
        print(f'  Annotated frame saved -> {frame_path}')
    print(BORDER)
    print()


def print_safe_heartbeat(environment: str, frame_index: int) -> None:
    ts = datetime.datetime.utcnow().strftime('%H:%M:%S')
    print(f'  [{ts}] SAFE  | env={environment.upper():10s} | frame={frame_index}')


print('Console alert formatter loaded.')

Console alert formatter loaded.


In [ ]:
def process_pipeline_result(
    result: Dict[str, Any],
    config: Dict = CONFIG,
) -> Optional[Dict]:
    """
    Main integration function. Call once per processed frame.
    Returns the logged event record (dict) if UNSAFE, else None.
    """
    safety_label     = str(result.get('safety_label', 'safe')).lower()
    environment      = str(result.get('environment',  'unknown')).lower()
    scene_desc       = str(result.get('scene_description', ''))
    confidence       = float(result.get('confidence', 0.0))
    detected_objects = result.get('detected_objects',   [])
    scene_nodes      = result.get('scene_graph_nodes',  [])
    scene_edges      = result.get('scene_graph_edges',  [])
    contrib_nodes    = result.get('contributing_nodes', [])
    raw_frame        = result.get('frame', None)
    frame_index      = int(result.get('frame_index', -1))
    video_source     = str(result.get('video_source', 'unknown'))
    session_id       = str(result.get('session_id',   'default_session'))
    event_label_raw  = result.get('event_label', '')

    # Route SAFE scenes
    if safety_label != 'unsafe':
        if config.get('log_safe_scenes', False):
            print_safe_heartbeat(environment, frame_index)
        return None

    # Confidence gate
    if confidence < config['alert_confidence_threshold']:
        return None

    # Resolve event label
    event_label = (
        str(event_label_raw) if event_label_raw
        else (infer_event_label_from_objects(detected_objects, environment) or 'Unsafe Event')
    )

    # Build reasoning
    threat_reasoning = build_threat_reasoning(
        environment, detected_objects, event_label, scene_desc
    )

    # Save annotated frame
    temp_id = hashlib.md5(f'{session_id}-{time.time()}'.encode()).hexdigest()[:12]
    frame_path = _frame_annotator.annotate_and_save(
        raw_frame, detected_objects, event_label, environment,
        confidence, temp_id, config['frames_dir'], config['jpeg_quality']
    )

    # Write event log
    event_record = _event_logger.write(
        session_id, environment, event_label, scene_desc, confidence,
        detected_objects, scene_nodes, scene_edges, contrib_nodes,
        frame_path, video_source, frame_index, threat_reasoning
    )

    # Print console alert
    print_unsafe_alert(
        event_record['event_id'], event_record['timestamp_iso'],
        environment, event_label, confidence, threat_reasoning, scene_desc,
        detected_objects, frame_path, video_source, frame_index
    )

    return event_record


print('process_pipeline_result() ready -- this is the integration point.')

process_pipeline_result() ready -- this is the integration point.


## Self- Test with Mock Data

In [ ]:
print('Running self-test with mock pipeline results...')
print()

# Test 1: Gun in an office (UNSAFE)
r1 = process_pipeline_result({
    'safety_label'     : 'unsafe',
    'environment'      : 'office',
    'event_label'      : 'Gun Detection',
    'scene_description': (
        'A person is standing near a desk in a corporate office. '
        'The individual appears to be holding a handgun pointed toward another person. '
        'Multiple monitors and office furniture are visible in the background.'
    ),
    'confidence'       : 0.91,
    'detected_objects' : [
        {'label': 'Person', 'confidence': 0.97, 'bounding_box': [45,  80,  220, 460]},
        {'label': 'Gun',    'confidence': 0.91, 'bounding_box': [160, 210, 240, 270]},
        {'label': 'Person', 'confidence': 0.88, 'bounding_box': [310, 100, 520, 480]},
    ],
    'scene_graph_nodes' : [
        {'id': 'n1', 'label': 'Person', 'attributes': []},
        {'id': 'n2', 'label': 'Gun',    'attributes': []},
        {'id': 'n3', 'label': 'Person', 'attributes': []},
    ],
    'scene_graph_edges' : [
        {'source': 'n1', 'target': 'n2', 'relation': 'holding_or_wielding'},
        {'source': 'n1', 'target': 'n3', 'relation': 'adjacent_to'},
    ],
    'contributing_nodes': ['n1', 'n2'],
    'frame'            : None,
    'frame_index'      : 150,
    'video_source'     : 'office_cam_01.mp4',
    'session_id'       : 'test_session_001',
})

# Test 2: Fire in a classroom (UNSAFE)
r2 = process_pipeline_result({
    'safety_label'     : 'unsafe',
    'environment'      : 'classroom',
    'event_label'      : 'Fire Detection',
    'scene_description': (
        'The classroom appears to be on fire near the front of the room. '
        'Flames are visible near the desk and thick smoke fills the upper frame. '
        'Several students appear to be evacuating.'
    ),
    'confidence'       : 0.85,
    'detected_objects' : [
        {'label': 'Fire',   'confidence': 0.85, 'bounding_box': [100, 320, 380, 480]},
        {'label': 'Smoke',  'confidence': 0.72, 'bounding_box': [0,   0,   640, 200]},
        {'label': 'Person', 'confidence': 0.90, 'bounding_box': [480, 100, 620, 470]},
    ],
    'scene_graph_nodes': [
        {'id': 'n1', 'label': 'Fire',   'attributes': ['active']},
        {'id': 'n2', 'label': 'Smoke',  'attributes': ['spreading']},
        {'id': 'n3', 'label': 'Person', 'attributes': ['evacuating']},
    ],
    'scene_graph_edges': [
        {'source': 'n1', 'target': 'n2', 'relation': 'engulfing_or_spreading_to'},
        {'source': 'n1', 'target': 'n3', 'relation': 'reacting_to_or_fleeing'},
    ],
    'contributing_nodes': ['n1', 'n2'],
    'frame'            : None,
    'frame_index'      : 275,
    'video_source'     : 'classroom_cam_02.mp4',
    'session_id'       : 'test_session_001',
})

# Test 3: SAFE scene — should return None
r3 = process_pipeline_result({'safety_label':'safe','environment':'home','frame_index':30,'session_id':'test_session_001'})
assert r3 is None
print('SAFE scene correctly returned None (no alert).')

# Test 4: Fall — event_label blank, auto-inferred from objects
r4 = process_pipeline_result({
    'safety_label'     : 'unsafe',
    'environment'      : 'home',
    'event_label'      : '',
    'scene_description': (
        'An elderly person appears to have fallen on the kitchen floor and is not moving. '
        'The individual is lying near the counter.'
    ),
    'confidence'       : 0.78,
    'detected_objects' : [
        {'label': 'Person Falling', 'confidence': 0.78, 'bounding_box': [80, 300, 400, 470]},
    ],
    'scene_graph_nodes': [{'id': 'n1', 'label': 'Person', 'attributes': ['falling']}],
    'scene_graph_edges': [],
    'contributing_nodes': ['n1'],
    'frame'            : None,
    'frame_index'      : 90,
    'video_source'     : 'home_cam_01.mp4',
    'session_id'       : 'test_session_001',
})

print()
print('Self-test complete. Check ./alert_output/ for logs and frames.')

Running self-test with mock pipeline results...


UNSAFE SCENE DETECTED  --  GUN DETECTION
  Event ID     : aa34416a48f9
  Timestamp    : 2026-04-02T15:58:07.826396Z
  Environment  : OFFICE
  Threat Type  : Gun Detection
  Confidence   : 91.0%
  Video Source : office_cam_01.mp4
  Frame Index  : 150
----------------------------------------------------------------------
  DETECTED OBJECTS:
    * Person                  conf=97%  bbox=[45,80,220,460]
    * Gun                     conf=91%  bbox=[160,210,240,270]
    * Person                  conf=88%  bbox=[310,100,520,480]
----------------------------------------------------------------------
  THREAT REASONING:
    Environment: OFFICE
    Detected objects: Person (97%), Gun (91%), Person (88%)
    Threat type: Gun Detection
    Rule applied: Firearms are never permitted in a OFFICE environment.
    Scene context: A person is standing near a desk in a corporate office. The individual appears to be holding a handgun pointed toward another

/tmp/ipykernel_9646/3799085566.py:30: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  iso = datetime.datetime.utcfromtimestamp(now).strftime('%Y-%m-%dT%H:%M:%S.%f') + 'Z'


## Log Inspection Utility

In [ ]:
def load_event_log(log_file: str = CONFIG['log_file']) -> List[Dict]:
    events = []
    if not Path(log_file).exists():
        print(f'No log file at {log_file}')
        return events
    with open(log_file, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                events.append(json.loads(line))
    return events


def print_event_summary(events: List[Dict]) -> None:
    if not events:
        print('No events logged yet.')
        return
    print(f'\n{chr(8212)*70}')
    print(f'  EVENT LOG SUMMARY  ({len(events)} event(s))')
    print(f'{chr(8212)*70}')
    print(f"  {'EVENT ID':14s}  {'TIMESTAMP (UTC)':22s}  {'ENV':10s}  {'LABEL':18s}  CONF")
    print(f"  {'-'*12}  {'-'*20}  {'-'*8}  {'-'*16}  ----")
    for ev in events:
        ts = ev.get('timestamp_iso', '')[:19].replace('T', ' ')
        print(
            f"  {ev.get('event_id',''):14s}  {ts:22s}  "
            f"{ev.get('environment',''):10s}  "
            f"{ev.get('event_label',''):18s}  "
            f"{ev.get('confidence',0):.0%}"
        )
    print(f'{chr(8212)*70}\n')


events = load_event_log()
print_event_summary(events)

if events:
    print('Sample record (first entry):')
    s = events[0].copy()
    s['scene_description'] = s['scene_description'][:80] + '...'
    s['threat_reasoning']  = s['threat_reasoning'][:80]  + '...'
    print(json.dumps(s, indent=2))


——————————————————————————————————————————————————————————————————————
  EVENT LOG SUMMARY  (3 event(s))
——————————————————————————————————————————————————————————————————————
  EVENT ID        TIMESTAMP (UTC)         ENV         LABEL               CONF
  ------------  --------------------  --------  ----------------  ----
  aa34416a48f9    2026-04-02 15:58:07     office      Gun Detection       91%
  8e7dc588d430    2026-04-02 15:58:07     classroom   Fire Detection      85%
  048921aa5434    2026-04-02 15:58:07     home        Fall Detection      78%
——————————————————————————————————————————————————————————————————————

Sample record (first entry):
{
  "event_id": "aa34416a48f9",
  "session_id": "test_session_001",
  "timestamp_iso": "2026-04-02T15:58:07.826396Z",
  "timestamp_epoch": 1775145487.8263962,
  "environment": "office",
  "event_label": "Gun Detection",
  "scene_description": "A person is standing near a desk in a corporate office. The individual appears t...",
  "confi

In [ ]:
EXPORT_PATH = './alert_Generator.ipynb'

src = '''
import json, time, datetime, hashlib
from pathlib import Path
from typing import Optional, Dict, Any, List

try:
    import cv2, numpy as np
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False

CONFIG = {
    "output_dir": "./alert_output",
    "log_file": "./alert_output/safety_events.log",
    "frames_dir": "./alert_output/frames",
    "alert_confidence_threshold": 0.20,
    "log_safe_scenes": False,
    "jpeg_quality": 90,
}
Path(CONFIG["output_dir"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["frames_dir"]).mkdir(parents=True, exist_ok=True)

ENVIRONMENT_RULES = {
    "home":      ["gun","knife","fire","flames","smoke","burning","person falling","fall"],
    "office":    ["gun","fire","flames","smoke","person falling","fall"],
    "classroom": ["gun","knife","fire","flames","smoke","burning","person falling","fall"],
}
EVENT_LABEL_MAP = {
    "gun":"Gun Detection","knife":"Knife Detection",
    "fire":"Fire Detection","flames":"Fire Detection",
    "smoke":"Fire Detection","burning":"Fire Detection",
    "person falling":"Fall Detection","fall":"Fall Detection",
}
BORDER, THIN = "="*70, "-"*70


class EventLogger:
    def __init__(self, lf): self.log_file = lf
    def _eid(self, ts, sid): return hashlib.md5(f"{sid}-{ts}".encode()).hexdigest()[:12]
    def write(self, session_id, environment, event_label, scene_description, confidence,
              detected_objects, scene_graph_nodes, scene_graph_edges, contributing_nodes,
              frame_path, video_source, frame_index, threat_reasoning):
        now=time.time()
        iso=datetime.datetime.utcfromtimestamp(now).strftime("%Y-%m-%dT%H:%M:%S.%f")+"Z"
        rec={"event_id":self._eid(now,session_id),"session_id":session_id,"timestamp_iso":iso,
             "timestamp_epoch":now,"environment":environment,"event_label":event_label,
             "scene_description":scene_description,"confidence":round(confidence,4),
             "detected_objects":detected_objects,"scene_graph_nodes":scene_graph_nodes,
             "scene_graph_edges":scene_graph_edges,"contributing_nodes":contributing_nodes,
             "frame_path":frame_path,"video_source":video_source,
             "frame_index":frame_index,"threat_reasoning":threat_reasoning}
        with open(self.log_file,"a",encoding="utf-8") as f: f.write(json.dumps(rec)+"\\n")
        return rec


class FrameAnnotator:
    COLOURS={"gun":(0,0,255),"knife":(0,128,255),"fire":(0,69,255),"fall":(255,0,255),"person":(0,255,0),"default":(255,255,0)}
    def _col(self,l): return next((v for k,v in self.COLOURS.items() if k in l.lower()),self.COLOURS["default"])
    def annotate_and_save(self,frame,objs,elabel,env,conf,eid,fdir,jq=90):
        if not CV2_AVAILABLE: return None
        if frame is None: frame=np.zeros((480,640,3),dtype=np.uint8)
        ann=frame.copy(); h,w=ann.shape[:2]
        for obj in objs:
            box=obj.get("bounding_box"); lbl=obj.get("label","?"); cf=obj.get("confidence",0)
            if box and len(box)==4:
                x1,y1,x2,y2=[int(v) for v in box]; c=self._col(lbl)
                cv2.rectangle(ann,(x1,y1),(x2,y2),c,2)
                tag=f"{lbl} {cf:.0%}"; (tw,th),_=cv2.getTextSize(tag,cv2.FONT_HERSHEY_SIMPLEX,0.55,1)
                cv2.rectangle(ann,(x1,y1-th-6),(x1+tw+4,y1),c,-1)
                cv2.putText(ann,tag,(x1+2,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1)
        cv2.rectangle(ann,(0,h-50),(w,h),(0,0,180),-1)
        cv2.putText(ann,f"UNSAFE|{elabel}|{env.upper()}|{conf:.0%}|{eid}",(8,h-16),cv2.FONT_HERSHEY_SIMPLEX,0.50,(255,255,255),1)
        p=str(Path(fdir)/f"{eid}.jpg")
        cv2.imwrite(p,ann,[cv2.IMWRITE_JPEG_QUALITY,jq])
        return p


def build_threat_reasoning(env,objs,elabel,desc):
    obj_s=", ".join(f\"{o[\"label\"]} ({o.get(\"confidence\",0):.0%})\" for o in objs) or "none"
    note={"Gun Detection":f"Firearms not permitted in {env.upper()}.",
          "Knife Detection":"Knife outside kitchen - threat." if env=="home" else f"Bladed weapons not permitted in {env.upper()}.",
          "Fire Detection":f"Fire/flames/smoke in {env.upper()} - hazard.",
          "Fall Detection":"Person fallen - may need help."}.get(elabel,f"Unsafe in {env.upper()}.")
    return f"Environment: {env.upper()}\\nObjects: {obj_s}\\nThreat: {elabel}\\nRule: {note}\\nContext: {desc.strip()}"


def infer_event_label_from_objects(objs,env):
    for obj in objs:
        ll=obj.get("label","").lower()
        for t in ENVIRONMENT_RULES.get(env.lower(),[]):
            if t in ll: return EVENT_LABEL_MAP.get(t,"Unsafe Event")
    return None


def print_unsafe_alert(eid,ts,env,elabel,conf,reasoning,desc,objs,fpath,vsrc,fidx):
    print(); print(BORDER); print(f"UNSAFE SCENE DETECTED -- {elabel.upper()}"); print(BORDER)
    print(f"  Event ID    : {eid}"); print(f"  Timestamp   : {ts}")
    print(f"  Environment : {env.upper()}"); print(f"  Confidence  : {conf:.1%}")
    print(f"  Source      : {vsrc}  Frame: {fidx}"); print(THIN)
    print("  DETECTED OBJECTS:")
    for obj in objs:
        box=obj.get("bounding_box"); bbox=f"[{box[0]:.0f},{box[1]:.0f},{box[2]:.0f},{box[3]:.0f}]" if box else "N/A"
        print(f\"    * {obj.get(\"label\",\"?\"):22s}  conf={obj.get(\"confidence\",0):.0%}  bbox={bbox}\")
    print(THIN+"\\n  THREAT REASONING:")
    for ln in reasoning.splitlines(): print(f"    {ln}")
    print(THIN+"\\n  SCENE DESCRIPTION:")
    d=desc.strip()
    for i in range(0,max(len(d),1),65): print(f"    {d[i:i+65]}")
    if fpath: print(f"\\n  Frame saved -> {fpath}")
    print(BORDER); print()


_logger    = EventLogger(CONFIG["log_file"])
_annotator = FrameAnnotator()


def process_pipeline_result(result, config=CONFIG):
    """Main integration function. Call once per processed frame."""
    sl=str(result.get("safety_label","safe")).lower()
    env=str(result.get("environment","unknown")).lower()
    desc=str(result.get("scene_description",""))
    conf=float(result.get("confidence",0.0))
    objs=result.get("detected_objects",[])
    nodes=result.get("scene_graph_nodes",[])
    edges=result.get("scene_graph_edges",[])
    cnodes=result.get("contributing_nodes",[])
    frame=result.get("frame",None)
    fidx=int(result.get("frame_index",-1))
    vsrc=str(result.get("video_source","unknown"))
    sid=str(result.get("session_id","default_session"))
    el_raw=result.get("event_label","")
    if sl != "unsafe": return None
    if conf < config["alert_confidence_threshold"]: return None
    el=str(el_raw) if el_raw else (infer_event_label_from_objects(objs,env) or "Unsafe Event")
    reasoning=build_threat_reasoning(env,objs,el,desc)
    tid=hashlib.md5(f"{sid}-{time.time()}".encode()).hexdigest()[:12]
    fp=_annotator.annotate_and_save(frame,objs,el,env,conf,tid,config["frames_dir"],config["jpeg_quality"])
    rec=_logger.write(sid,env,el,desc,conf,objs,nodes,edges,cnodes,fp,vsrc,fidx,reasoning)
    print_unsafe_alert(rec["event_id"],rec["timestamp_iso"],env,el,conf,reasoning,desc,objs,fp,vsrc,fidx)
    return rec
'''

with open(EXPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(src.strip())

print(f'alert_generator.py exported -> {EXPORT_PATH}')
print('In your sgg-model notebook, add:')
print('    from alert_generator import process_pipeline_result')

alert_generator.py exported -> ./alert_Generator.ipynb
In your sgg-model notebook, add:
    from alert_generator import process_pipeline_result
